# 🎓 Day 2 · Session 2
# 02B · RAGAS Hands-on
## Evaluate RAG systems using faithfulness, relevancy, precision and recall

## 🎯 Learning Objectives

- Build a small RAG evaluation pipeline
- Create a Golden Test Set
- Calculate RAGAS-style metrics
- Compare retrieval settings

## 🔧 Setup Note

These notebooks are designed for **VS Code, Jupyter and Google Colab**.

For API-based evaluation demos, keep your `.env` file in the same folder as the notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas numpy

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
import numpy as np
from openai import OpenAI

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully.")
print("Key starts with:", api_key[:10], "...")

In [ ]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.2, system_prompt="You are a helpful AI evaluation assistant.", max_tokens=800):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ RAGAS Hands-on

RAGAS evaluates RAG systems using:
- Faithfulness
- Answer Relevancy
- Context Precision
- Context Recall

This notebook demonstrates the concept with a lightweight classroom-friendly implementation.

In [ ]:
# Optional full RAGAS install:
# %pip install ragas datasets langchain langchain-openai

# 2️⃣ Sample Knowledge Base

In [ ]:
knowledge_base = {
    "department_handbook.txt": '''
M.Tech CSE tuition fee is Rs. 50,000 per semester.
The programme duration is four semesters.
GATE-qualified students receive a 50% tuition scholarship.
Students must maintain 75% attendance.
''',
    "course_catalog.txt": '''
The NLP elective is offered in Semester 3.
The faculty coordinator is Dr. Kavitha Raman.
Prerequisites for NLP are Machine Learning and Python Programming.
''',
    "lab_manual.txt": '''
AI lab records must be submitted every Friday.
The labs include Python for AI, preprocessing, regression, classification, neural networks and model evaluation.
'''
}

# 3️⃣ Simple Retriever

In [ ]:
def simple_retrieve(question, top_k=2):
    question_terms = set(question.lower().replace("?", "").split())
    scored = []
    for source, text in knowledge_base.items():
        text_terms = set(text.lower().split())
        overlap = len(question_terms.intersection(text_terms))
        scored.append({"source": source, "context": text, "score": overlap})
    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]

simple_retrieve("Who teaches NLP?", top_k=2)

# 4️⃣ RAG Answer Function

In [ ]:
def rag_answer(question, top_k=2):
    retrieved = simple_retrieve(question, top_k=top_k)
    context = "\n\n".join([f"Source: {r['source']}\n{r['context']}" for r in retrieved])
    prompt = f'''
You are a careful academic assistant.
Answer using ONLY the context below.
If the answer is not present, say: "I don't have this information in the provided knowledge base."
Mention the source.

Context:
{context}

Question:
{question}
'''
    answer = ask_openai(prompt)
    return {"question": question, "answer": answer, "contexts": [r["context"] for r in retrieved], "sources": [r["source"] for r in retrieved]}

rag_answer("Who teaches NLP?")

# 5️⃣ Golden Test Set

In [ ]:
golden_test_set = [
    {"question": "What is the M.Tech CSE fee per semester?", "ground_truth": "Rs. 50,000 per semester", "expected_source": "department_handbook.txt"},
    {"question": "Who teaches NLP?", "ground_truth": "Dr. Kavitha Raman", "expected_source": "course_catalog.txt"},
    {"question": "What are the prerequisites for NLP?", "ground_truth": "Machine Learning and Python Programming", "expected_source": "course_catalog.txt"},
    {"question": "When should AI lab records be submitted?", "ground_truth": "Every Friday", "expected_source": "lab_manual.txt"},
    {"question": "What is the hostel fee?", "ground_truth": "Not available", "expected_source": "none"},
]
pd.DataFrame(golden_test_set)

# 6️⃣ Run RAG on Golden Set

In [ ]:
eval_rows = []
for item in golden_test_set:
    rag_result = rag_answer(item["question"], top_k=2)
    eval_rows.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "expected_source": item["expected_source"],
        "answer": rag_result["answer"],
        "retrieved_sources": rag_result["sources"]
    })

eval_df = pd.DataFrame(eval_rows)
eval_df

# 7️⃣ Simple Metric Approximation

In [ ]:
def contains_ground_truth(answer, ground_truth):
    if ground_truth.lower() == "not available":
        return "don't have" in answer.lower() or "not available" in answer.lower()
    parts = [p.strip().lower() for p in ground_truth.split(" and ")]
    return any(p in answer.lower() for p in parts)

def source_precision(retrieved_sources, expected_source):
    if expected_source == "none":
        return 1.0
    return 1.0 if expected_source in retrieved_sources else 0.0

scores = []
for _, row in eval_df.iterrows():
    answer_ok = contains_ground_truth(row["answer"], row["ground_truth"])
    source_ok = source_precision(row["retrieved_sources"], row["expected_source"])
    scores.append({
        "question": row["question"],
        "faithfulness": 1.0 if answer_ok else 0.5,
        "answer_relevancy": 1.0 if answer_ok else 0.0,
        "context_precision": source_ok,
        "context_recall": source_ok,
    })

score_df = pd.DataFrame(scores)
score_df

# 8️⃣ Evaluation Dashboard

In [ ]:
dashboard = pd.DataFrame({
    "Metric": ["Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall"],
    "Score": [
        score_df["faithfulness"].mean(),
        score_df["answer_relevancy"].mean(),
        score_df["context_precision"].mean(),
        score_df["context_recall"].mean()
    ]
})
dashboard

# 9️⃣ A/B Test top_k

In [ ]:
def evaluate_top_k(k):
    rows = []
    for item in golden_test_set:
        rag_result = rag_answer(item["question"], top_k=k)
        answer_ok = contains_ground_truth(rag_result["answer"], item["ground_truth"])
        source_ok = source_precision(rag_result["sources"], item["expected_source"])
        rows.append({
            "faithfulness": 1.0 if answer_ok else 0.5,
            "answer_relevancy": 1.0 if answer_ok else 0.0,
            "context_precision": source_ok,
            "context_recall": source_ok,
        })
    df = pd.DataFrame(rows)
    return {"top_k": k, "faithfulness": df["faithfulness"].mean(), "answer_relevancy": df["answer_relevancy"].mean(), "context_precision": df["context_precision"].mean(), "context_recall": df["context_recall"].mean()}

pd.DataFrame([evaluate_top_k(1), evaluate_top_k(2), evaluate_top_k(3)])

# 🧪 Faculty Lab

Add 5 more questions, include out-of-scope questions, change top_k, and compare scores.

# ✅ Summary

RAGAS-style metrics identify whether the issue is retrieval, context quality, or answer generation.